# QM8 Baseline Comparison (7 Modelle)

Dieses Notebook vergleicht auf **demselben Split** sieben Baselines fuer **QM8 Multi-Task Regression (16 Targets)**:
1. RF auf Graph-Features (task-wise)
2. RF auf RDKit-Deskriptoren (task-wise)
3. ChebNet von Scratch (task-wise)
4. ChebNet mit Foundation-Checkpoint (task-wise 2-stage FT)
5. Standard-SEG (echtes Multi-Task Regression)
6. SEG: Foundation-ChebNet direkt kopiert -> end-to-end Fine-Tuning
7. SEG: Foundation -> Cheb QM8-Mix FT -> SEG -> end-to-end Fine-Tuning

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

import deepchem as dc
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

here = Path.cwd().resolve()
workspace_root = next((p for p in [here, *here.parents] if (p / 'models').exists()), here)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from models.cheb_predictor import ChebPredictor, ChebPredictorConfig
from models.seg_predictor import SEGPredictor, SEGPredictorConfig
from utils.embedding_cache import EfficientEmbeddingCache
from utils.molecular_graph import smiles_to_graph

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

print(f'Workspace root: {workspace_root}')
print(f'Device: {device}')

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Workspace root: C:\Users\robsc\Home\Dev\molfusion2
Device: cuda


In [2]:
def set_all_seeds(s):
    s = int(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def untransform_labels(y, transformers):
    out = np.asarray(y, dtype=np.float32).copy()
    for transformer in reversed(transformers):
        out = transformer.untransform(out)
    return np.asarray(out, dtype=np.float32)

def taskwise_regression_metrics(y_true, y_pred, observed_mask):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    observed_mask = np.asarray(observed_mask, dtype=bool)

    maes, rmses, r2s = [], [], []
    n_tasks = y_true.shape[1]

    for t in range(n_tasks):
        m = observed_mask[:, t] & np.isfinite(y_true[:, t]) & np.isfinite(y_pred[:, t])
        if m.sum() == 0:
            maes.append(np.nan)
            rmses.append(np.nan)
            r2s.append(np.nan)
            continue

        yt = y_true[m, t]
        yp = y_pred[m, t]
        maes.append(float(mean_absolute_error(yt, yp)))
        rmses.append(float(np.sqrt(mean_squared_error(yt, yp))))
        if len(np.unique(yt)) >= 2:
            r2s.append(float(r2_score(yt, yp)))
        else:
            r2s.append(np.nan)

    maes_arr = np.asarray(maes, dtype=np.float32)
    rmses_arr = np.asarray(rmses, dtype=np.float32)
    r2_arr = np.asarray(r2s, dtype=np.float32)

    return {
        'macro_mae': float(np.nanmean(maes_arr)) if np.isfinite(np.nanmean(maes_arr)) else np.nan,
        'macro_rmse': float(np.nanmean(rmses_arr)) if np.isfinite(np.nanmean(rmses_arr)) else np.nan,
        'macro_r2': float(np.nanmean(r2_arr)) if np.isfinite(np.nanmean(r2_arr)) else np.nan,
        'n_valid_tasks': int(np.sum(np.isfinite(maes_arr))),
        'per_task_mae': maes,
        'per_task_rmse': rmses,
        'per_task_r2': r2s,
    }

In [3]:
# QM8 with scaffold split + label normalization (DeepChem default transformers).
tasks, datasets, transformers = dc.molnet.load_qm8(featurizer='ECFP', splitter='scaffold')
train_dc, valid_dc, test_dc = datasets

task_names = list(tasks)
n_tasks = len(task_names)

train_smiles = list(train_dc.ids)
valid_smiles = list(valid_dc.ids)
test_smiles = list(test_dc.ids)

train_y_norm = np.asarray(train_dc.y, dtype=np.float32)
valid_y_norm = np.asarray(valid_dc.y, dtype=np.float32)
test_y_norm = np.asarray(test_dc.y, dtype=np.float32)

# QM8 usually has no missing labels, but keep finite-mask handling generic.
train_mask = np.isfinite(train_y_norm)
valid_mask = np.isfinite(valid_y_norm)
test_mask = np.isfinite(test_y_norm)

train_y_nan = train_y_norm.copy()
valid_y_nan = valid_y_norm.copy()
test_y_nan = test_y_norm.copy()
train_y_nan[~train_mask] = np.nan
valid_y_nan[~valid_mask] = np.nan
test_y_nan[~test_mask] = np.nan

train_y_orig = untransform_labels(train_y_norm, transformers)
valid_y_orig = untransform_labels(valid_y_norm, transformers)
test_y_orig = untransform_labels(test_y_norm, transformers)

print(f'QM8 tasks: {n_tasks}')
print(f'Split sizes -> train={len(train_smiles)}, val={len(valid_smiles)}, test={len(test_smiles)}')
print(f'Observed labels -> train={int(train_mask.sum())}, val={int(valid_mask.sum())}, test={int(test_mask.sum())}')
print(f'Transformers: {[type(t).__name__ for t in transformers]}')

QM8 tasks: 16
Split sizes -> train=17397, val=2175, test=2175
Observed labels -> train=278352, val=34800, test=34800
Transformers: ['NormalizationTransformer']


In [4]:
def _impute_with_train_median(train_X, valid_X, test_X):
    med = np.nanmedian(train_X, axis=0)
    med = np.where(np.isfinite(med), med, 0.0).astype(np.float32)

    def _imp(X):
        X = np.asarray(X, dtype=np.float32)
        mask = ~np.isfinite(X)
        if mask.any():
            X = X.copy()
            X[mask] = np.take(med, np.where(mask)[1])
        return X

    return _imp(train_X), _imp(valid_X), _imp(test_X)

def graph_feature_vector_from_smiles(smiles):
    g = smiles_to_graph(smiles, add_hydrogens=False)
    X = np.asarray(g.X, dtype=np.float32)
    ew = np.asarray(g.edge_weight, dtype=np.float32)

    x_sum = X.sum(axis=0)
    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0)

    n_nodes = float(g.n_nodes)
    n_edges_undir = float(g.edge_index.shape[1] // 2)
    ew_mean = float(ew.mean()) if ew.size else 0.0
    ew_std = float(ew.std()) if ew.size else 0.0
    ew_sum = float(ew.sum()) if ew.size else 0.0

    return np.concatenate([
        x_sum, x_mean, x_std,
        np.array([n_nodes, n_edges_undir, ew_mean, ew_std, ew_sum], dtype=np.float32),
    ])

def build_graph_feature_matrix(smiles_list):
    rows = []
    for smi in smiles_list:
        try:
            rows.append(graph_feature_vector_from_smiles(smi))
        except Exception:
            rows.append(np.full((89,), np.nan, dtype=np.float32))
    return np.vstack(rows).astype(np.float32)

def rdkit_descriptor_matrix(smiles_list):
    desc_names = [name for name, _ in Descriptors.descList]
    n_desc = len(desc_names)
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append(np.full((n_desc,), np.nan, dtype=np.float32))
            continue
        all_descs = Descriptors.CalcMolDescriptors(mol)
        row = []
        for name in desc_names:
            try:
                row.append(float(all_descs.get(name, np.nan)))
            except Exception:
                row.append(np.nan)
        rows.append(np.asarray(row, dtype=np.float32))
    return np.vstack(rows).astype(np.float32), desc_names

In [5]:
# Feature matrices for RF baselines.
g_train = build_graph_feature_matrix(train_smiles)
g_valid = build_graph_feature_matrix(valid_smiles)
g_test = build_graph_feature_matrix(test_smiles)
g_train, g_valid, g_test = _impute_with_train_median(g_train, g_valid, g_test)

d_train, desc_names = rdkit_descriptor_matrix(train_smiles)
d_valid, _ = rdkit_descriptor_matrix(valid_smiles)
d_test, _ = rdkit_descriptor_matrix(test_smiles)
d_train, d_valid, d_test = _impute_with_train_median(d_train, d_valid, d_test)

print('Graph feature dim:', g_train.shape[1])
print('RDKit descriptor dim:', d_train.shape[1])

Graph feature dim: 89
RDKit descriptor dim: 217


In [6]:
CHEB_CFG = {
    'hidden_channels': 128,
    'K': 3,
    'num_layers': 3,
    'pool': 'sum',
}

VANILLA_FIT_KWARGS = {
    'batch_size': 64,
    'num_epochs': 120,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'patience': 20,
}

CHEB_FIT_KWARGS = {
    'batch_size': 64,
    'frozen_epochs': 30,
    'unfreeze_epochs': 90,
    'head_lr': 5e-4,
    'encoder_lr': 5e-5,
    'weight_decay': 1e-1,
    'patience': 20,
}

SEG_FIT_KWARGS = {
    'hidden_channels': 128,
    'K': 3,
    'num_layers': 3,
    'pool': 'sum',
    'set2set_processing_steps': 6,
    'fusion': 'cross_mha',
    'fusion_dim': 64,
    'fusion_n_heads': 8,
    'text_proj_init': 'xavier',
    'text_proj_init_gain': 0.1,
    'freeze_text_proj': True,
    'dropout': 0.3,
    'fusion_dropout': 0.3,
    'head_dropout': 0.6,
    'weight_decay': 1e-1,
    'head_type': 'mlp',
    'head_hidden_dim': 64,
    'learning_rate': 3e-4,
    'batch_size': 64,
    'num_epochs': 120,
    'patience': 20,
    'scheduler': 'cosine',
    'scheduler_patience': 5,
    'scheduler_factor': 0.5,
    'min_lr': 1e-6,
    'grad_clip': None,
}

SEG_CHEB_INIT_FIT_KWARGS = {**SEG_FIT_KWARGS, **{
    'hidden_channels': CHEB_CFG['hidden_channels'],
    'K': CHEB_CFG['K'],
    'num_layers': CHEB_CFG['num_layers'],
    'pool': CHEB_CFG['pool'],
    'learning_rate': 2e-4,
}}

SEG_TRANSFER_FIT_KWARGS = {
    'batch_size': 64,
    'frozen_epochs': 25,
    'unfreeze_epochs': 75,
    'fusion_lr': 2e-4,
    'encoder_lr': 5e-5,
    'weight_decay': 1e-1,
    'patience': 15,
}

def make_qm8_cheb_config():
    return ChebPredictorConfig(
        task='regression',
        hidden_channels=CHEB_CFG['hidden_channels'],
        K=CHEB_CFG['K'],
        num_layers=CHEB_CFG['num_layers'],
        pool=CHEB_CFG['pool'],
        dropout=0.3,
        lambda_max=2.0,
        add_hydrogens=False,
        head_hidden_dim=32,
        head_dropout=0.5,
    )

def make_seg_config(text_embedding_dim):
    return SEGPredictorConfig(
        task='regression',
        num_tasks=n_tasks,
        hidden_channels=SEG_FIT_KWARGS['hidden_channels'],
        K=SEG_FIT_KWARGS['K'],
        num_layers=SEG_FIT_KWARGS['num_layers'],
        dropout=SEG_FIT_KWARGS['dropout'],
        pool=SEG_FIT_KWARGS['pool'],
        set2set_processing_steps=SEG_FIT_KWARGS['set2set_processing_steps'],
        text_embedding_dim=int(text_embedding_dim),
        text_projection_dim=SEG_FIT_KWARGS['fusion_dim'],
        text_proj_init=SEG_FIT_KWARGS['text_proj_init'],
        text_proj_init_gain=SEG_FIT_KWARGS['text_proj_init_gain'],
        freeze_text_proj=SEG_FIT_KWARGS['freeze_text_proj'],
        fusion=SEG_FIT_KWARGS['fusion'],
        fusion_dim=SEG_FIT_KWARGS['fusion_dim'],
        fusion_n_heads=SEG_FIT_KWARGS['fusion_n_heads'],
        fusion_dropout=SEG_FIT_KWARGS['fusion_dropout'],
        head_type=SEG_FIT_KWARGS['head_type'],
        head_hidden_dim=SEG_FIT_KWARGS['head_hidden_dim'],
        head_dropout=SEG_FIT_KWARGS['head_dropout'],
    )

def make_seg_pretrained_graph_config(text_embedding_dim):
    return SEGPredictorConfig(
        task='regression',
        num_tasks=n_tasks,
        hidden_channels=SEG_CHEB_INIT_FIT_KWARGS['hidden_channels'],
        K=SEG_CHEB_INIT_FIT_KWARGS['K'],
        num_layers=SEG_CHEB_INIT_FIT_KWARGS['num_layers'],
        dropout=SEG_CHEB_INIT_FIT_KWARGS['dropout'],
        pool=SEG_CHEB_INIT_FIT_KWARGS['pool'],
        set2set_processing_steps=SEG_CHEB_INIT_FIT_KWARGS['set2set_processing_steps'],
        text_embedding_dim=int(text_embedding_dim),
        text_projection_dim=SEG_CHEB_INIT_FIT_KWARGS['fusion_dim'],
        text_proj_init=SEG_CHEB_INIT_FIT_KWARGS['text_proj_init'],
        text_proj_init_gain=SEG_CHEB_INIT_FIT_KWARGS['text_proj_init_gain'],
        freeze_text_proj=SEG_CHEB_INIT_FIT_KWARGS['freeze_text_proj'],
        fusion=SEG_CHEB_INIT_FIT_KWARGS['fusion'],
        fusion_dim=SEG_CHEB_INIT_FIT_KWARGS['fusion_dim'],
        fusion_n_heads=SEG_CHEB_INIT_FIT_KWARGS['fusion_n_heads'],
        fusion_dropout=SEG_CHEB_INIT_FIT_KWARGS['fusion_dropout'],
        head_type=SEG_CHEB_INIT_FIT_KWARGS['head_type'],
        head_hidden_dim=SEG_CHEB_INIT_FIT_KWARGS['head_hidden_dim'],
        head_dropout=SEG_CHEB_INIT_FIT_KWARGS['head_dropout'],
    )

In [7]:
def load_encoder_from_pretraining(predictor, pretrain_checkpoint_path):
    ckpt = torch.load(pretrain_checkpoint_path, map_location=predictor.device, weights_only=False)
    ckpt_cfg = ckpt.get('cheb_cfg', {})
    required_keys = ('hidden_channels', 'K', 'num_layers', 'pool')
    mismatches = []
    for k in required_keys:
        if k in ckpt_cfg and ckpt_cfg[k] != CHEB_CFG[k]:
            mismatches.append((k, ckpt_cfg[k], CHEB_CFG[k]))
    if mismatches:
        mismatch_msg = '; '.join([f"{k}: ckpt={a} current={b}" for (k, a, b) in mismatches])
        raise RuntimeError(f'Checkpoint encoder config mismatch: {mismatch_msg}')

    pre_state = ckpt['model_state_dict']
    enc_state = {k.replace('encoder.', '', 1): v for k, v in pre_state.items() if k.startswith('encoder.')}
    predictor._encoder.load_state_dict(enc_state, strict=True)

    pool_state = {k.replace('pooling.', '', 1): v for k, v in pre_state.items() if k.startswith('pooling.')}
    if pool_state:
        try:
            predictor._pooling.load_state_dict(pool_state, strict=True)
            print('Loaded pooling weights from pretraining checkpoint.')
        except Exception as exc:
            print(f'Pooling weights not loaded (shape or module mismatch): {exc}')

def make_cheb_backbone_from_pretraining(pretrain_checkpoint_path, sample_smiles):
    cheb_backbone = ChebPredictor(config=make_qm8_cheb_config(), device=str(device))
    _, in_channels = cheb_backbone._precompute_graphs(sample_smiles, verbose=False)
    cheb_backbone._in_channels = in_channels
    cheb_backbone._build_model()
    load_encoder_from_pretraining(cheb_backbone, pretrain_checkpoint_path)
    cheb_backbone._is_fitted = True
    return cheb_backbone

def fit_two_stage(
    predictor,
    train_smiles, train_y,
    valid_smiles, valid_y,
    *,
    seed=42,
    batch_size=32,
    frozen_epochs=40,
    unfreeze_epochs=160,
    head_lr=5e-4,
    encoder_lr=5e-5,
    weight_decay=1e-1,
    patience=30,
    pretrained_ckpt=None,
    verbose=True,
):
    train_graphs, in_channels = predictor._precompute_graphs(train_smiles, verbose=verbose)
    valid_graphs, _ = predictor._precompute_graphs(valid_smiles, verbose=verbose)
    predictor._in_channels = in_channels
    predictor._build_model()

    if pretrained_ckpt is not None:
        load_encoder_from_pretraining(predictor, pretrained_ckpt)

    y_train = np.asarray(train_y, dtype=np.float32)
    y_valid = np.asarray(valid_y, dtype=np.float32)
    train_idx = np.arange(len(train_smiles))
    valid_idx = list(range(len(valid_smiles)))
    loss_fn = nn.MSELoss()

    rng = np.random.default_rng(seed)
    history = {'train_loss': [], 'val_loss': [], 'stage': []}
    best_val = float('inf')
    best_state = None
    stale = 0

    for p in predictor._encoder.parameters():
        p.requires_grad = False
    for p in predictor._pooling.parameters():
        p.requires_grad = False
    for p in predictor._head.parameters():
        p.requires_grad = True

    opt_head = torch.optim.Adam(
        list(predictor._head.parameters()),
        lr=head_lr,
        weight_decay=weight_decay,
    )

    for _ in range(1, frozen_epochs + 1):
        tr = predictor._train_epoch(train_graphs, y_train, train_idx, batch_size, opt_head, loss_fn, rng)
        vl = predictor._evaluate(valid_graphs, y_valid, valid_idx, batch_size, loss_fn)
        history['train_loss'].append(float(tr))
        history['val_loss'].append(float(vl))
        history['stage'].append('frozen')
        if vl < best_val:
            best_val = float(vl)
            best_state = predictor._get_state_dict()
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    for p in predictor._encoder.parameters():
        p.requires_grad = True
    for p in predictor._pooling.parameters():
        p.requires_grad = True
    for p in predictor._head.parameters():
        p.requires_grad = True

    opt_full = torch.optim.Adam(
        [
            {'params': list(predictor._encoder.parameters()) + list(predictor._pooling.parameters()), 'lr': encoder_lr},
            {'params': list(predictor._head.parameters()), 'lr': head_lr},
        ],
        weight_decay=weight_decay,
    )

    stale = 0
    for _ in range(1, unfreeze_epochs + 1):
        tr = predictor._train_epoch(train_graphs, y_train, train_idx, batch_size, opt_full, loss_fn, rng)
        vl = predictor._evaluate(valid_graphs, y_valid, valid_idx, batch_size, loss_fn)
        history['train_loss'].append(float(tr))
        history['val_loss'].append(float(vl))
        history['stage'].append('unfreeze')
        if vl < best_val:
            best_val = float(vl)
            best_state = predictor._get_state_dict()
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        predictor._load_state_dict(best_state)
    predictor._is_fitted = True
    return history, best_val

def fit_seg_two_stage(
    predictor,
    train_smiles, train_y,
    valid_smiles, valid_y,
    *,
    train_text_embeddings,
    valid_text_embeddings,
    seed=42,
    batch_size=64,
    frozen_epochs=25,
    unfreeze_epochs=75,
    fusion_lr=2e-4,
    encoder_lr=5e-5,
    weight_decay=1e-1,
    patience=15,
    verbose=True,
):
    if predictor._encoder is None or predictor._fusion is None or predictor._head is None:
        train_graphs, in_channels = predictor._precompute_graphs(train_smiles, verbose=verbose)
        valid_graphs, _ = predictor._precompute_graphs(valid_smiles, verbose=verbose)
        predictor._in_channels = in_channels
        predictor._build_model()
    else:
        train_graphs, _ = predictor._precompute_graphs(train_smiles, verbose=verbose)
        valid_graphs, _ = predictor._precompute_graphs(valid_smiles, verbose=verbose)

    if isinstance(train_text_embeddings, torch.Tensor):
        train_text_emb = train_text_embeddings.detach().cpu().numpy()
    else:
        train_text_emb = np.asarray(train_text_embeddings, dtype=np.float32)

    if isinstance(valid_text_embeddings, torch.Tensor):
        valid_text_emb = valid_text_embeddings.detach().cpu().numpy()
    else:
        valid_text_emb = np.asarray(valid_text_embeddings, dtype=np.float32)

    y_train = np.asarray(train_y, dtype=np.float32)
    y_valid = np.asarray(valid_y, dtype=np.float32)
    train_idx = np.arange(len(train_smiles))
    valid_idx = list(range(len(valid_smiles)))

    is_multitask = (predictor.config.num_tasks > 1)
    loss_fn = nn.MSELoss(reduction='none') if is_multitask else nn.MSELoss()

    rng = np.random.default_rng(seed)
    history = {'train_loss': [], 'val_loss': [], 'stage': []}
    best_val = float('inf')
    best_state = None
    stale = 0

    for p in predictor._encoder.parameters():
        p.requires_grad = False
    for p in predictor._pooling.parameters():
        p.requires_grad = False
    for p in predictor._fusion.parameters():
        p.requires_grad = True
    for p in predictor._head.parameters():
        p.requires_grad = True

    stage1_params = list(predictor._fusion.parameters()) + list(predictor._head.parameters())
    if predictor._text_proj is not None:
        for p in predictor._text_proj.parameters():
            p.requires_grad = True
        stage1_params += list(predictor._text_proj.parameters())

    opt_stage1 = torch.optim.AdamW(stage1_params, lr=fusion_lr, weight_decay=weight_decay)

    for _ in range(1, frozen_epochs + 1):
        tr = predictor._train_epoch(
            train_graphs, train_text_emb, y_train, train_idx, batch_size, opt_stage1, loss_fn, rng,
            is_multilabel=is_multitask,
        )
        vl = predictor._evaluate(
            valid_graphs, valid_text_emb, y_valid, valid_idx, batch_size, loss_fn,
            is_multilabel=is_multitask,
        )
        history['train_loss'].append(float(tr))
        history['val_loss'].append(float(vl))
        history['stage'].append('frozen')
        if vl < best_val:
            best_val = float(vl)
            best_state = predictor._get_state_dict()
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    for p in predictor._encoder.parameters():
        p.requires_grad = True
    for p in predictor._pooling.parameters():
        p.requires_grad = True
    for p in predictor._fusion.parameters():
        p.requires_grad = True
    for p in predictor._head.parameters():
        p.requires_grad = True

    graph_params = list(predictor._encoder.parameters()) + list(predictor._pooling.parameters())
    fusion_head_params = list(predictor._fusion.parameters()) + list(predictor._head.parameters())
    if predictor._text_proj is not None:
        fusion_head_params += list(predictor._text_proj.parameters())

    opt_stage2 = torch.optim.AdamW(
        [
            {'params': graph_params, 'lr': encoder_lr},
            {'params': fusion_head_params, 'lr': fusion_lr},
        ],
        weight_decay=weight_decay,
    )

    stale = 0
    for _ in range(1, unfreeze_epochs + 1):
        tr = predictor._train_epoch(
            train_graphs, train_text_emb, y_train, train_idx, batch_size, opt_stage2, loss_fn, rng,
            is_multilabel=is_multitask,
        )
        vl = predictor._evaluate(
            valid_graphs, valid_text_emb, y_valid, valid_idx, batch_size, loss_fn,
            is_multilabel=is_multitask,
        )
        history['train_loss'].append(float(tr))
        history['val_loss'].append(float(vl))
        history['stage'].append('unfreeze')
        if vl < best_val:
            best_val = float(vl)
            best_state = predictor._get_state_dict()
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        predictor._load_state_dict(best_state)
    predictor._is_fitted = True
    return history, best_val

REQUIRED_PRETRAIN_STAGE = 'stage2_descriptor'

def resolve_config_specific_checkpoint(workspace_root, cheb_cfg, required_stage=REQUIRED_PRETRAIN_STAGE):
    ckpt_dir = workspace_root / 'cache' / 'chemeleon_pretraining'
    if not ckpt_dir.exists():
        raise FileNotFoundError(f'Checkpoint directory not found: {ckpt_dir}')

    patterns = [
        (
            f"cheb_foundation_2stage__h{cheb_cfg['hidden_channels']}__"
            f"L{cheb_cfg['num_layers']}__"
            f"K{cheb_cfg['K']}__*.pt"
        ),
        (
            f"cheb_foundation__h{cheb_cfg['hidden_channels']}__"
            f"L{cheb_cfg['num_layers']}__"
            f"K{cheb_cfg['K']}__*.pt"
        ),
    ]

    candidates = []
    for pat in patterns:
        for p in ckpt_dir.glob(pat):
            if p.name == 'cheb_foundation_pretrain.pt' or '__fallback__' in p.name:
                continue
            candidates.append(p)

    uniq = []
    seen = set()
    for p in candidates:
        s = str(p.resolve())
        if s not in seen:
            uniq.append(p)
            seen.add(s)
    candidates = uniq

    ranked = []
    for p in candidates:
        score = float('inf')
        stage_ok = False
        stage_val = None
        try:
            payload = torch.load(p, map_location='cpu', weights_only=False)
            if isinstance(payload, dict):
                raw = payload.get('best_val_masked_mse', None)
                if raw is not None:
                    score = float(raw)
                stage_val = payload.get('pretraining_stage', None)
                stage_ok = (stage_val == required_stage) if required_stage is not None else True
        except Exception:
            pass
        ranked.append((0 if stage_ok else 1, score, -p.stat().st_mtime, p, stage_val))

    if ranked:
        ranked.sort(key=lambda t: (t[0], t[1], t[2]))
        _, _, _, best_path, best_stage = ranked[0]
        print(f'Checkpoint stage metadata: {best_stage}')
        return best_path

    alias = ckpt_dir / 'cheb_foundation_pretrain.pt'
    if alias.exists():
        payload = torch.load(alias, map_location='cpu', weights_only=False)
        alias_stage = payload.get('pretraining_stage', None) if isinstance(payload, dict) else None
        if required_stage is not None and alias_stage != required_stage:
            raise RuntimeError(
                f"Alias checkpoint exists but pretraining_stage={alias_stage!r} does not match required stage {required_stage!r}."
            )
        return alias

    raise FileNotFoundError(
        f'No matching config-specific checkpoint found in {ckpt_dir} for required stage {required_stage!r}'
    )

In [8]:
def fit_rf_taskwise_regression(X_train, y_train, m_train, X_valid, y_valid, m_valid, X_test, y_test, m_test, *, seed=42):
    n_tasks = y_train.shape[1]
    valid_pred = np.full((len(X_valid), n_tasks), np.nan, dtype=np.float32)
    test_pred = np.full((len(X_test), n_tasks), np.nan, dtype=np.float32)

    for t in range(n_tasks):
        tr_mask = m_train[:, t]
        if tr_mask.sum() < 20:
            continue

        reg = RandomForestRegressor(
            n_estimators=800,
            max_features='sqrt',
            random_state=int(seed),
            n_jobs=-1,
        )
        reg.fit(X_train[tr_mask], y_train[tr_mask, t])
        valid_pred[:, t] = reg.predict(X_valid)
        test_pred[:, t] = reg.predict(X_test)

    return valid_pred, test_pred

def build_mixed_task_regression_dataset(smiles_list, y, m):
    xs, ys = [], []
    for i, smi in enumerate(smiles_list):
        for t in range(y.shape[1]):
            if not m[i, t]:
                continue
            xs.append(smi)
            ys.append(float(y[i, t]))
    return xs, ys

def fit_cheb_taskwise_regression(
    train_smiles, train_y, train_m,
    valid_smiles, valid_y, valid_m,
    test_smiles, test_y, test_m,
    *,
    seed=42,
    pretrained_ckpt=None,
    use_two_stage=False,
    verbose=False,
):
    n_tasks = train_y.shape[1]
    valid_pred = np.full((len(valid_smiles), n_tasks), np.nan, dtype=np.float32)
    test_pred = np.full((len(test_smiles), n_tasks), np.nan, dtype=np.float32)
    task_best_vals = []

    for t in range(n_tasks):
        tr_mask = train_m[:, t]
        va_mask = valid_m[:, t]
        if tr_mask.sum() < 20 or va_mask.sum() < 20:
            task_best_vals.append(np.nan)
            continue

        ytr = train_y[tr_mask, t]
        yva = valid_y[va_mask, t]
        tr_sm = [train_smiles[i] for i in np.where(tr_mask)[0].tolist()]
        va_sm = [valid_smiles[i] for i in np.where(va_mask)[0].tolist()]

        model = ChebPredictor(config=make_qm8_cheb_config(), device=str(device))
        if use_two_stage:
            _, best_val = fit_two_stage(
                model, tr_sm, ytr.tolist(), va_sm, yva.tolist(),
                seed=int(seed),
                batch_size=CHEB_FIT_KWARGS['batch_size'],
                frozen_epochs=CHEB_FIT_KWARGS['frozen_epochs'],
                unfreeze_epochs=CHEB_FIT_KWARGS['unfreeze_epochs'],
                head_lr=CHEB_FIT_KWARGS['head_lr'],
                encoder_lr=CHEB_FIT_KWARGS['encoder_lr'],
                weight_decay=CHEB_FIT_KWARGS['weight_decay'],
                patience=CHEB_FIT_KWARGS['patience'],
                pretrained_ckpt=pretrained_ckpt,
                verbose=verbose,
            )
        else:
            hist = model.fit(
                tr_sm, ytr.tolist(),
                val_smiles=va_sm, val_labels=yva.tolist(),
                num_epochs=VANILLA_FIT_KWARGS['num_epochs'],
                batch_size=VANILLA_FIT_KWARGS['batch_size'],
                learning_rate=VANILLA_FIT_KWARGS['learning_rate'],
                weight_decay=VANILLA_FIT_KWARGS['weight_decay'],
                patience=VANILLA_FIT_KWARGS['patience'],
                seed=int(seed),
                verbose=verbose,
            )
            best_val = float(np.min(hist['val_loss'])) if 'val_loss' in hist and len(hist['val_loss']) > 0 else np.nan

        task_best_vals.append(float(best_val))
        valid_pred[:, t] = model.predict_batch(valid_smiles)
        test_pred[:, t] = model.predict_batch(test_smiles)

    best_val_mean = float(np.nanmean(np.asarray(task_best_vals, dtype=np.float32))) if np.isfinite(np.nanmean(np.asarray(task_best_vals, dtype=np.float32))) else np.nan
    return valid_pred, test_pred, best_val_mean

In [9]:
# --- Checkpoint + embeddings ---
pretrained_ckpt = resolve_config_specific_checkpoint(workspace_root, CHEB_CFG)
print('Using pretrained checkpoint:', pretrained_ckpt.resolve())

emb_dir = workspace_root / 'cache' / 'cot_embeddings'
npz_path = emb_dir / 'quantum_fast_text_embeddings_compact.npz'
if not npz_path.exists():
    raise FileNotFoundError(f'SEG embedding cache not found: {npz_path}')

seg_cache = EfficientEmbeddingCache.load(npz_path)
all_smiles_all = train_smiles + valid_smiles + test_smiles
all_emb = torch.from_numpy(seg_cache.get_batch(all_smiles_all)).float()
n_train = len(train_smiles)
n_valid = len(valid_smiles)
train_emb = all_emb[:n_train]
valid_emb = all_emb[n_train:n_train + n_valid]
test_emb = all_emb[n_train + n_valid:]

# --- Baseline 1: RF graph features (task-wise) ---
rf_graph_valid_pred_norm, rf_graph_test_pred_norm = fit_rf_taskwise_regression(
    g_train, train_y_norm, train_mask,
    g_valid, valid_y_norm, valid_mask,
    g_test, test_y_norm, test_mask,
    seed=seed,
)

# --- Baseline 2: RF RDKit descriptors (task-wise) ---
rf_desc_valid_pred_norm, rf_desc_test_pred_norm = fit_rf_taskwise_regression(
    d_train, train_y_norm, train_mask,
    d_valid, valid_y_norm, valid_mask,
    d_test, test_y_norm, test_mask,
    seed=seed,
)

# --- Baseline 3: Cheb scratch (task-wise) ---
cheb_scratch_valid_pred_norm, cheb_scratch_test_pred_norm, cheb_scratch_best_val = fit_cheb_taskwise_regression(
    train_smiles, train_y_norm, train_mask,
    valid_smiles, valid_y_norm, valid_mask,
    test_smiles, test_y_norm, test_mask,
    seed=seed,
    pretrained_ckpt=None,
    use_two_stage=False,
    verbose=False,
)

# --- Baseline 4: Cheb pretrained (task-wise 2-stage) ---
cheb_pre_valid_pred_norm, cheb_pre_test_pred_norm, cheb_pre_best_val = fit_cheb_taskwise_regression(
    train_smiles, train_y_norm, train_mask,
    valid_smiles, valid_y_norm, valid_mask,
    test_smiles, test_y_norm, test_mask,
    seed=seed,
    pretrained_ckpt=pretrained_ckpt,
    use_two_stage=True,
    verbose=False,
)

# --- Baseline 5: SEG standard (true multi-task regression) ---
seg = SEGPredictor(config=make_seg_config(text_embedding_dim=all_emb.shape[1]), device=str(device))
seg_hist = seg.fit(
    smiles_list=train_smiles,
    labels=train_y_nan,
    val_smiles=valid_smiles,
    val_labels=valid_y_nan,
    text_embeddings=train_emb,
    val_text_embeddings=valid_emb,
    num_epochs=SEG_FIT_KWARGS['num_epochs'],
    batch_size=SEG_FIT_KWARGS['batch_size'],
    learning_rate=SEG_FIT_KWARGS['learning_rate'],
    weight_decay=SEG_FIT_KWARGS['weight_decay'],
    patience=SEG_FIT_KWARGS['patience'],
    scheduler=SEG_FIT_KWARGS['scheduler'],
    scheduler_patience=SEG_FIT_KWARGS['scheduler_patience'],
    scheduler_factor=SEG_FIT_KWARGS['scheduler_factor'],
    min_lr=SEG_FIT_KWARGS['min_lr'],
    grad_clip=SEG_FIT_KWARGS['grad_clip'],
    seed=seed,
    verbose=True,
)
seg_best_val = float(np.nanmin(seg_hist['val_loss']))
seg_valid_pred_norm = seg.predict_batch(valid_smiles, text_embeddings=valid_emb)
seg_test_pred_norm = seg.predict_batch(test_smiles, text_embeddings=test_emb)

# --- Baseline 6: Foundation-ChebNet -> SEG two-stage ---
cheb_foundation_backbone = make_cheb_backbone_from_pretraining(pretrained_ckpt, train_smiles)
seg_foundation_e2e = SEGPredictor(
    config=make_seg_pretrained_graph_config(text_embedding_dim=all_emb.shape[1]),
    device=str(device),
)
seg_foundation_e2e.init_from_cheb(cheb_foundation_backbone)
_, seg_foundation_e2e_best_val = fit_seg_two_stage(
    seg_foundation_e2e,
    train_smiles, train_y_nan,
    valid_smiles, valid_y_nan,
    train_text_embeddings=train_emb,
    valid_text_embeddings=valid_emb,
    seed=seed,
    batch_size=SEG_TRANSFER_FIT_KWARGS['batch_size'],
    frozen_epochs=SEG_TRANSFER_FIT_KWARGS['frozen_epochs'],
    unfreeze_epochs=SEG_TRANSFER_FIT_KWARGS['unfreeze_epochs'],
    fusion_lr=SEG_TRANSFER_FIT_KWARGS['fusion_lr'],
    encoder_lr=SEG_TRANSFER_FIT_KWARGS['encoder_lr'],
    weight_decay=SEG_TRANSFER_FIT_KWARGS['weight_decay'],
    patience=SEG_TRANSFER_FIT_KWARGS['patience'],
    verbose=True,
)
seg_foundation_valid_pred_norm = seg_foundation_e2e.predict_batch(valid_smiles, text_embeddings=valid_emb)
seg_foundation_test_pred_norm = seg_foundation_e2e.predict_batch(test_smiles, text_embeddings=test_emb)

# --- Baseline 7: Foundation -> Cheb QM8-Mix FT -> SEG two-stage ---
mix_train_smiles, mix_train_labels = build_mixed_task_regression_dataset(train_smiles, train_y_norm, train_mask)
mix_valid_smiles, mix_valid_labels = build_mixed_task_regression_dataset(valid_smiles, valid_y_norm, valid_mask)

cheb_qm8_adapted = ChebPredictor(config=make_qm8_cheb_config(), device=str(device))
_, _ = fit_two_stage(
    cheb_qm8_adapted,
    mix_train_smiles, mix_train_labels,
    mix_valid_smiles, mix_valid_labels,
    seed=seed,
    batch_size=CHEB_FIT_KWARGS['batch_size'],
    frozen_epochs=CHEB_FIT_KWARGS['frozen_epochs'],
    unfreeze_epochs=CHEB_FIT_KWARGS['unfreeze_epochs'],
    head_lr=CHEB_FIT_KWARGS['head_lr'],
    encoder_lr=CHEB_FIT_KWARGS['encoder_lr'],
    weight_decay=CHEB_FIT_KWARGS['weight_decay'],
    patience=CHEB_FIT_KWARGS['patience'],
    pretrained_ckpt=pretrained_ckpt,
    verbose=True,
)

seg_qm8_adapted_e2e = SEGPredictor(
    config=make_seg_pretrained_graph_config(text_embedding_dim=all_emb.shape[1]),
    device=str(device),
)
seg_qm8_adapted_e2e.init_from_cheb(cheb_qm8_adapted)
_, seg_qm8_adapted_e2e_best_val = fit_seg_two_stage(
    seg_qm8_adapted_e2e,
    train_smiles, train_y_nan,
    valid_smiles, valid_y_nan,
    train_text_embeddings=train_emb,
    valid_text_embeddings=valid_emb,
    seed=seed,
    batch_size=SEG_TRANSFER_FIT_KWARGS['batch_size'],
    frozen_epochs=SEG_TRANSFER_FIT_KWARGS['frozen_epochs'],
    unfreeze_epochs=SEG_TRANSFER_FIT_KWARGS['unfreeze_epochs'],
    fusion_lr=SEG_TRANSFER_FIT_KWARGS['fusion_lr'],
    encoder_lr=SEG_TRANSFER_FIT_KWARGS['encoder_lr'],
    weight_decay=SEG_TRANSFER_FIT_KWARGS['weight_decay'],
    patience=SEG_TRANSFER_FIT_KWARGS['patience'],
    verbose=True,
)
seg_qm8_adapted_valid_pred_norm = seg_qm8_adapted_e2e.predict_batch(valid_smiles, text_embeddings=valid_emb)
seg_qm8_adapted_test_pred_norm = seg_qm8_adapted_e2e.predict_batch(test_smiles, text_embeddings=test_emb)

# Convert all predictions back to original scale before scoring.
rf_graph_test_pred = untransform_labels(rf_graph_test_pred_norm, transformers)
rf_desc_test_pred = untransform_labels(rf_desc_test_pred_norm, transformers)
cheb_scratch_test_pred = untransform_labels(cheb_scratch_test_pred_norm, transformers)
cheb_pre_test_pred = untransform_labels(cheb_pre_test_pred_norm, transformers)
seg_test_pred = untransform_labels(seg_test_pred_norm, transformers)
seg_foundation_test_pred = untransform_labels(seg_foundation_test_pred_norm, transformers)
seg_qm8_adapted_test_pred = untransform_labels(seg_qm8_adapted_test_pred_norm, transformers)

rf_graph_metrics = taskwise_regression_metrics(test_y_orig, rf_graph_test_pred, test_mask)
rf_desc_metrics = taskwise_regression_metrics(test_y_orig, rf_desc_test_pred, test_mask)
cheb_scratch_metrics = taskwise_regression_metrics(test_y_orig, cheb_scratch_test_pred, test_mask)
cheb_pre_metrics = taskwise_regression_metrics(test_y_orig, cheb_pre_test_pred, test_mask)
seg_metrics = taskwise_regression_metrics(test_y_orig, seg_test_pred, test_mask)
seg_foundation_metrics = taskwise_regression_metrics(test_y_orig, seg_foundation_test_pred, test_mask)
seg_qm8_adapted_metrics = taskwise_regression_metrics(test_y_orig, seg_qm8_adapted_test_pred, test_mask)

Checkpoint stage metadata: stage2_descriptor
Using pretrained checkpoint: C:\Users\robsc\Home\Dev\molfusion2\cache\chemeleon_pretraining\cheb_foundation_2stage__h128__L3__K3__dim-1590__subset-200000__h-57e834e1e4.pt
Loading embeddings from quantum_fast_text_embeddings_compact.npz...
  Loaded 21722 entries, dim=3072
  Memory mode: mapped


KeyboardInterrupt: 

In [ ]:
comparison_df = pd.DataFrame([
    {'model': 'rf_graph_features', 'n_valid_tasks': rf_graph_metrics['n_valid_tasks'], 'best_val_loss': np.nan, **{k:v for k,v in rf_graph_metrics.items() if k.startswith('macro_')}},
    {'model': 'rf_rdkit_descriptors', 'n_valid_tasks': rf_desc_metrics['n_valid_tasks'], 'best_val_loss': np.nan, **{k:v for k,v in rf_desc_metrics.items() if k.startswith('macro_')}},
    {'model': 'cheb_scratch_taskwise', 'n_valid_tasks': cheb_scratch_metrics['n_valid_tasks'], 'best_val_loss': cheb_scratch_best_val, **{k:v for k,v in cheb_scratch_metrics.items() if k.startswith('macro_')}},
    {'model': 'cheb_pretrained_taskwise', 'n_valid_tasks': cheb_pre_metrics['n_valid_tasks'], 'best_val_loss': cheb_pre_best_val, **{k:v for k,v in cheb_pre_metrics.items() if k.startswith('macro_')}},
    {'model': 'seg_standard_multitask', 'n_valid_tasks': seg_metrics['n_valid_tasks'], 'best_val_loss': seg_best_val, **{k:v for k,v in seg_metrics.items() if k.startswith('macro_')}},
    {'model': 'seg_foundation_e2e', 'n_valid_tasks': seg_foundation_metrics['n_valid_tasks'], 'best_val_loss': seg_foundation_e2e_best_val, **{k:v for k,v in seg_foundation_metrics.items() if k.startswith('macro_')}},
    {'model': 'seg_qm8_adapted_e2e', 'n_valid_tasks': seg_qm8_adapted_metrics['n_valid_tasks'], 'best_val_loss': seg_qm8_adapted_e2e_best_val, **{k:v for k,v in seg_qm8_adapted_metrics.items() if k.startswith('macro_')}},
])

comparison_df = comparison_df.sort_values('macro_mae', ascending=True).reset_index(drop=True)
display(comparison_df)

plot_df = comparison_df.copy()

fig, axes = plt.subplots(3, 1, figsize=(12, 11), constrained_layout=True)
palette = plt.cm.Set2(np.linspace(0.08, 0.92, len(plot_df)))

# Lower is better for MAE and RMSE
for ax, metric_name, title in [
    (axes[0], 'macro_mae', 'Macro MAE (lower is better)'),
    (axes[1], 'macro_rmse', 'Macro RMSE (lower is better)'),
]:
    vals = plot_df[metric_name].values
    y = np.arange(len(plot_df))
    bars = ax.barh(y, vals, color=palette, edgecolor='black', linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['model'])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.25)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

# Higher is better for R2
vals = plot_df['macro_r2'].values
y = np.arange(len(plot_df))
bars = axes[2].barh(y, vals, color=palette, edgecolor='black', linewidth=0.3)
axes[2].set_yticks(y)
axes[2].set_yticklabels(plot_df['model'])
axes[2].invert_yaxis()
axes[2].set_title('Macro R2 (higher is better)')
axes[2].grid(axis='x', alpha=0.25)
for bar, v in zip(bars, vals):
    axes[2].text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

plt.show()

out_csv = workspace_root / 'benchmarking' / 'results' / 'qm8_baseline_comparison.csv'
comparison_df.to_csv(out_csv, index=False)
print(f'Saved comparison CSV to: {out_csv}')